In [1]:
# ============================================================
# CELL 1 — IMPORTS & CONFIGURATION
# Franchise Performance Benchmarking & Outlier Intelligence
# ============================================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os
import warnings
warnings.filterwarnings('ignore')

# Seed for reproducibility — every run gives same dataset
np.random.seed(42)
random.seed(42)

print("✅ Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

✅ Libraries imported successfully
Pandas version: 2.3.3
NumPy version: 2.2.6


In [2]:
# ============================================================
# CELL 2 — DEFINE OUTLETS, CITIES & FRANCHISE METADATA
# ============================================================

# 12 Indian cities with tier classification
cities = {
    'Mumbai':      {'tier': 1, 'cost_index': 1.35, 'demand_index': 1.40},
    'Delhi':       {'tier': 1, 'cost_index': 1.30, 'demand_index': 1.35},
    'Bangalore':   {'tier': 1, 'cost_index': 1.25, 'demand_index': 1.30},
    'Hyderabad':   {'tier': 1, 'cost_index': 1.20, 'demand_index': 1.25},
    'Pune':        {'tier': 2, 'cost_index': 1.10, 'demand_index': 1.15},
    'Chennai':     {'tier': 2, 'cost_index': 1.10, 'demand_index': 1.10},
    'Ahmedabad':   {'tier': 2, 'cost_index': 1.05, 'demand_index': 1.05},
    'Kolkata':     {'tier': 2, 'cost_index': 1.00, 'demand_index': 1.00},
    'Jaipur':      {'tier': 3, 'cost_index': 0.90, 'demand_index': 0.90},
    'Lucknow':     {'tier': 3, 'cost_index': 0.85, 'demand_index': 0.85},
    'Indore':      {'tier': 3, 'cost_index': 0.82, 'demand_index': 0.82},
    'Chandigarh':  {'tier': 3, 'cost_index': 0.88, 'demand_index': 0.88},
}

# Outlet performance archetypes
archetypes = {
    'star':       {'weight': 0.20, 'base_orders': 18, 'performance': 0.90},
    'stable':     {'weight': 0.35, 'base_orders': 12, 'performance': 0.72},
    'struggling': {'weight': 0.25, 'base_orders': 7,  'performance': 0.50},
    'critical':   {'weight': 0.12, 'base_orders': 4,  'performance': 0.30},
    'new':        {'weight': 0.08, 'base_orders': 9,  'performance': 0.60},
}

# Generate outlet IDs
outlet_ids = [f'OT{str(i).zfill(3)}' for i in range(1, 86)]

# Assign archetypes based on weights
archetype_list  = list(archetypes.keys())
archetype_weights = [archetypes[a]['weight'] for a in archetype_list]
assigned_archetypes = np.random.choice(archetype_list, size=85, p=archetype_weights)

# Assign cities — tier 1 cities get more outlets
city_list = list(cities.keys())
city_weights = np.array([
    0.12 if cities[c]['tier'] == 1
    else 0.09 if cities[c]['tier'] == 2
    else 0.06
    for c in city_list
])
city_weights = city_weights / city_weights.sum()
assigned_cities = np.random.choice(city_list, size=85, p=city_weights)

# Outlet open dates — spread across 5 years before data period
outlet_open_dates = [
    datetime(2023, 1, 1) - timedelta(days=int(np.random.randint(30, 1800)))
    for _ in range(85)
]

# Build outlet master table
outlet_master = pd.DataFrame({
    'outlet_id':              outlet_ids,
    'city':                   assigned_cities,
    'archetype':              assigned_archetypes,
    'tier':                   [cities[c]['tier'] for c in assigned_cities],
    'cost_index':             [cities[c]['cost_index'] for c in assigned_cities],
    'demand_index':           [cities[c]['demand_index'] for c in assigned_cities],
    'outlet_open_date':       outlet_open_dates,
    'outlet_age_months':      [int((datetime(2023,1,1) - d).days / 30) for d in outlet_open_dates],
    'seating_capacity':       np.random.choice([20, 30, 40, 50, 60], size=85),
    'franchise_fee_paid':     np.random.choice([True, False], size=85, p=[0.85, 0.15]),
    'zone':                   np.random.choice(['North', 'South', 'East', 'West', 'Central'], size=85),
    'outlet_type':            np.random.choice(['Standalone', 'Mall', 'Food Court', 'Highway'], size=85,
                                                p=[0.50, 0.25, 0.15, 0.10]),
})

print(f"✅ outlet_master created: {outlet_master.shape[0]} rows × {outlet_master.shape[1]} columns")
print(f"\nArchetype distribution:\n{outlet_master['archetype'].value_counts()}")
print(f"\nCity distribution:\n{outlet_master['city'].value_counts()}")
print(f"\nOutlet type distribution:\n{outlet_master['outlet_type'].value_counts()}")

✅ outlet_master created: 85 rows × 12 columns

Archetype distribution:
archetype
stable        26
star          25
struggling    17
critical       9
new            8
Name: count, dtype: int64

City distribution:
city
Bangalore     16
Mumbai        10
Pune           9
Kolkata        8
Hyderabad      7
Delhi          7
Indore         6
Ahmedabad      6
Chennai        5
Lucknow        4
Jaipur         4
Chandigarh     3
Name: count, dtype: int64

Outlet type distribution:
outlet_type
Standalone    36
Mall          19
Highway       18
Food Court    12
Name: count, dtype: int64


In [4]:
# ============================================================
# CELL 3 — GENERATE DAILY TRANSACTIONS (~4.5 LAKH ROWS)
# ============================================================

from datetime import date

# Date range: 1 Jan 2023 to 30 Jun 2024 (18 months = 547 days)
start_date = datetime(2023, 1, 1)
end_date   = datetime(2024, 6, 30)
date_range = pd.date_range(start=start_date, end=end_date, freq='D')

# Indian public holidays (simplified)
indian_holidays = [
    '2023-01-26', '2023-03-08', '2023-03-30', '2023-04-07',
    '2023-04-14', '2023-05-01', '2023-08-15', '2023-10-02',
    '2023-10-24', '2023-11-13', '2023-11-27', '2023-12-25',
    '2024-01-26', '2024-03-25', '2024-03-29', '2024-04-14',
    '2024-05-01', '2024-06-17',
]
indian_holidays = pd.to_datetime(indian_holidays)

# Order channels and their probabilities per archetype
channel_map = {
    'star':       {'Zomato': 0.35, 'Swiggy': 0.30, 'Direct App': 0.15, 'Dine-in': 0.15, 'WhatsApp': 0.05},
    'stable':     {'Zomato': 0.40, 'Swiggy': 0.30, 'Direct App': 0.10, 'Dine-in': 0.15, 'WhatsApp': 0.05},
    'struggling': {'Zomato': 0.50, 'Swiggy': 0.35, 'Direct App': 0.05, 'Dine-in': 0.08, 'WhatsApp': 0.02},
    'critical':   {'Zomato': 0.55, 'Swiggy': 0.38, 'Direct App': 0.02, 'Dine-in': 0.04, 'WhatsApp': 0.01},
    'new':        {'Zomato': 0.45, 'Swiggy': 0.35, 'Direct App': 0.08, 'Dine-in': 0.10, 'WhatsApp': 0.02},
}

# Item categories and base prices (₹)
item_categories = {
    'Burger':   {'base_price': 180, 'food_cost_pct': 0.32, 'packaging': 12},
    'Biryani':  {'base_price': 260, 'food_cost_pct': 0.35, 'packaging': 15},
    'Pizza':    {'base_price': 320, 'food_cost_pct': 0.30, 'packaging': 18},
    'Beverage': {'base_price': 80,  'food_cost_pct': 0.20, 'packaging': 8},
    'Dessert':  {'base_price': 120, 'food_cost_pct': 0.28, 'packaging': 10},
    'Combo':    {'base_price': 380, 'food_cost_pct': 0.33, 'packaging': 20},
}

# Payment methods
payment_methods = ['UPI', 'Card', 'Cash', 'Wallet']
payment_weights = [0.55, 0.25, 0.12, 0.08]

# Aggregator commission rates
commission_map = {
    'Zomato':     0.25,
    'Swiggy':     0.23,
    'Direct App': 0.05,
    'Dine-in':    0.00,
    'WhatsApp':   0.02,
}

print("⏳ Generating daily transactions... this will take 3-5 minutes")
print("Please wait, do not interrupt...\n")

all_transactions = []
transaction_counter = 1

for _, outlet in outlet_master.iterrows():
    outlet_id   = outlet['outlet_id']
    archetype   = outlet['archetype']
    cost_idx    = outlet['cost_index']
    demand_idx  = outlet['demand_index']
    base_orders = archetypes[archetype]['base_orders']
    performance = archetypes[archetype]['performance']
    channels    = channel_map[archetype]

    channel_names = list(channels.keys())
    channel_probs = list(channels.values())
    # Normalize channel probs
    channel_probs = np.array(channel_probs)
    channel_probs = channel_probs / channel_probs.sum()

    for single_date in date_range:
        dow         = single_date.dayofweek          # 0=Monday, 6=Sunday
        is_weekend  = 1 if dow >= 5 else 0
        is_holiday  = 1 if single_date in indian_holidays else 0

        # Demand multiplier — weekends and holidays boost orders
        demand_mult = demand_idx
        if is_weekend:
            demand_mult *= 1.25
        if is_holiday:
            demand_mult *= 1.40

        # Seasonal multiplier — Oct-Dec festive season boost
        month = single_date.month
        if month in [10, 11, 12]:
            demand_mult *= 1.15
        elif month in [5, 6]:
            demand_mult *= 0.90  # Summer slight dip

        # Number of orders that day for this outlet
        daily_orders = max(1, int(np.random.poisson(base_orders * demand_mult)))

        # Cap maximum orders realistically
        daily_orders = min(daily_orders, 35)

        # Kitchen utilization based on orders
        kitchen_util = min(0.99, daily_orders / 30)

        # Staff on shift
        staff_on_shift = max(2, int(np.random.normal(
            4 + performance * 4, 1
        )))

        for order_num in range(daily_orders):
            txn_id = f'TXN{str(transaction_counter).zfill(7)}'
            transaction_counter += 1

            # Order time — peak hours 12-2 PM and 7-10 PM
            hour_weights = np.array([
                1,1,1,1,1,1,        # 0-5 AM  — very low
                2,3,4,5,6,7,        # 6-11 AM — breakfast ramp
                10,12,10,8,7,8,     # 12-5 PM — lunch peak then dip
                9,12,14,12,10,6,    # 6-11 PM — dinner peak
            ], dtype=float)
            hour_weights /= hour_weights.sum()
            order_hour   = np.random.choice(range(24), p=hour_weights)
            order_minute = np.random.randint(0, 60)
            order_second = np.random.randint(0, 60)
            txn_time     = f"{str(order_hour).zfill(2)}:{str(order_minute).zfill(2)}:{str(order_second).zfill(2)}"

            # Channel
            channel = np.random.choice(channel_names, p=channel_probs)
            order_type = 'Dine-in' if channel == 'Dine-in' else 'Delivery' if channel in ['Zomato','Swiggy'] else 'Takeaway'

            # Item category
            item_cat = np.random.choice(list(item_categories.keys()),
                                         p=[0.25,0.20,0.18,0.15,0.10,0.12])
            item_info    = item_categories[item_cat]
            items_count  = np.random.choice([1,2,3,4], p=[0.40,0.35,0.18,0.07])

            # Gross amount with city cost index and slight noise
            base_price   = item_info['base_price'] * cost_idx * items_count
            gross_amount = round(base_price * np.random.uniform(0.90, 1.10), 2)

            # Discount — struggling and critical outlets offer more discounts
            discount_prob = {'star':0.15,'stable':0.20,'struggling':0.35,'critical':0.45,'new':0.30}
            if np.random.random() < discount_prob[archetype]:
                discount = round(gross_amount * np.random.uniform(0.05, 0.25), 2)
            else:
                discount = 0.0

            net_amount = round(gross_amount - discount, 2)

            # Costs
            food_cost        = round(net_amount * item_info['food_cost_pct'] * np.random.uniform(0.95,1.05), 2)
            packaging_cost   = round(item_info['packaging'] * items_count * cost_idx, 2)
            commission_rate  = commission_map[channel]
            agg_commission   = round(net_amount * commission_rate, 2)
            contribution_margin = round(net_amount - food_cost - packaging_cost - agg_commission, 2)

            # Delivery metrics
            if order_type == 'Delivery':
                delivery_time = max(15, int(np.random.normal(
                    35 - performance * 10, 8
                )))
                distance_km   = round(np.random.uniform(0.5, 8.0), 2)
                prep_time     = max(8, int(np.random.normal(18 - performance*5, 4)))
            else:
                delivery_time = None
                distance_km   = None
                prep_time     = max(5, int(np.random.normal(12, 3)))

            # Customer behaviour
            is_repeat     = 1 if np.random.random() < (0.25 + performance * 0.35) else 0
            is_new        = 1 - is_repeat
            upsell_taken  = 1 if np.random.random() < (0.15 + performance * 0.25) else 0

            # Rating — better performers get better ratings
            if np.random.random() < 0.55:
                rating_mean = 2.5 + performance * 2.0
                rating = round(min(5.0, max(1.0, np.random.normal(rating_mean, 0.5))), 1)
            else:
                rating = None

            # Complaint
            complaint_prob = max(0.01, 0.25 - performance * 0.20)
            had_complaint  = 1 if np.random.random() < complaint_prob else 0

            # Waste flag — critical outlets waste more
            waste_prob = max(0.05, 0.40 - performance * 0.30)
            waste_flag = 1 if np.random.random() < waste_prob else 0

            # Payment method
            payment = np.random.choice(payment_methods, p=payment_weights)

            # Customer ID — anonymised
            customer_id = f'CUST{str(np.random.randint(1, 50000)).zfill(6)}'

            all_transactions.append({
                'transaction_id':          txn_id,
                'outlet_id':               outlet_id,
                'transaction_date':        single_date.date(),
                'transaction_time':        txn_time,
                'day_of_week':             single_date.strftime('%A'),
                'month':                   single_date.strftime('%Y-%m'),
                'is_weekend':              is_weekend,
                'is_holiday':              is_holiday,
                'order_channel':           channel,
                'order_type':              order_type,
                'item_category':           item_cat,
                'items_ordered':           items_count,
                'gross_amount':            gross_amount,
                'discount_applied':        discount,
                'net_amount':              net_amount,
                'food_cost':               food_cost,
                'packaging_cost':          packaging_cost,
                'aggregator_commission':   agg_commission,
                'contribution_margin':     contribution_margin,
                'payment_method':          payment,
                'delivery_time_mins':      delivery_time,
                'prep_time_mins':          prep_time,
                'distance_km':             distance_km,
                'customer_id':             customer_id,
                'is_repeat_customer':      is_repeat,
                'is_new_customer':         is_new,
                'customer_rating':         rating,
                'had_complaint':           had_complaint,
                'upsell_taken':            upsell_taken,
                'staff_on_shift':          staff_on_shift,
                'kitchen_utilization_pct': round(kitchen_util, 2),
                'waste_flag':              waste_flag,
            })

# Convert to DataFrame
daily_transactions = pd.DataFrame(all_transactions)

print(f"✅ daily_transactions created: {daily_transactions.shape[0]:,} rows × {daily_transactions.shape[1]} columns")
print(f"\nDate range: {daily_transactions['transaction_date'].min()} to {daily_transactions['transaction_date'].max()}")
print(f"Total revenue generated: ₹{daily_transactions['net_amount'].sum():,.0f}")
print(f"Unique outlets: {daily_transactions['outlet_id'].nunique()}")
print(f"Unique customers: {daily_transactions['customer_id'].nunique()}")
print(f"\nOrder channel distribution:\n{daily_transactions['order_channel'].value_counts()}")

⏳ Generating daily transactions... this will take 3-5 minutes
Please wait, do not interrupt...

✅ daily_transactions created: 681,949 rows × 32 columns

Date range: 2023-01-01 to 2024-06-30
Total revenue generated: ₹328,771,804
Unique outlets: 85
Unique customers: 49999

Order channel distribution:
order_channel
Zomato        270578
Swiggy        212343
Dine-in        92342
Direct App     77126
WhatsApp       29560
Name: count, dtype: int64


In [5]:
# ============================================================
# CELL 4 — GENERATE MONTHLY PERFORMANCE TABLE
# Aggregated from daily_transactions (1,530 rows)
# ============================================================

monthly_perf = daily_transactions.groupby(['outlet_id', 'month']).agg(
    total_orders         = ('transaction_id',       'count'),
    revenue              = ('net_amount',            'sum'),
    gross_revenue        = ('gross_amount',          'sum'),
    total_discount       = ('discount_applied',      'sum'),
    total_food_cost      = ('food_cost',             'sum'),
    total_packaging_cost = ('packaging_cost',        'sum'),
    total_agg_commission = ('aggregator_commission', 'sum'),
    total_contribution   = ('contribution_margin',   'sum'),
    avg_order_value      = ('net_amount',            'mean'),
    avg_delivery_time    = ('delivery_time_mins',    'mean'),
    avg_prep_time        = ('prep_time_mins',        'mean'),
    avg_rating           = ('customer_rating',       'mean'),
    total_complaints     = ('had_complaint',         'sum'),
    total_waste_flags    = ('waste_flag',            'sum'),
    repeat_customers     = ('is_repeat_customer',    'sum'),
    new_customers        = ('is_new_customer',       'sum'),
    upsell_orders        = ('upsell_taken',          'sum'),
    is_weekend_orders    = ('is_weekend',            'sum'),
    is_holiday_orders    = ('is_holiday',            'sum'),
).reset_index()

# Derived KPI columns
monthly_perf['food_cost_pct']        = round((monthly_perf['total_food_cost']      / monthly_perf['revenue']) * 100, 2)
monthly_perf['packaging_cost_pct']   = round((monthly_perf['total_packaging_cost'] / monthly_perf['revenue']) * 100, 2)
monthly_perf['commission_cost_pct']  = round((monthly_perf['total_agg_commission'] / monthly_perf['revenue']) * 100, 2)
monthly_perf['discount_pct']         = round((monthly_perf['total_discount']       / monthly_perf['gross_revenue']) * 100, 2)
monthly_perf['contribution_margin_pct'] = round((monthly_perf['total_contribution']/ monthly_perf['revenue']) * 100, 2)
monthly_perf['complaint_rate']       = round((monthly_perf['total_complaints']     / monthly_perf['total_orders']) * 100, 2)
monthly_perf['waste_rate']           = round((monthly_perf['total_waste_flags']    / monthly_perf['total_orders']) * 100, 2)
monthly_perf['repeat_customer_pct']  = round((monthly_perf['repeat_customers']     / monthly_perf['total_orders']) * 100, 2)
monthly_perf['upsell_rate']          = round((monthly_perf['upsell_orders']        / monthly_perf['total_orders']) * 100, 2)
monthly_perf['avg_order_value']      = round(monthly_perf['avg_order_value'], 2)
monthly_perf['avg_rating']           = round(monthly_perf['avg_rating'], 2)
monthly_perf['avg_delivery_time']    = round(monthly_perf['avg_delivery_time'], 2)

# Add rental cost — fixed per outlet based on city tier
# Merge outlet master to get tier
monthly_perf = monthly_perf.merge(
    outlet_master[['outlet_id', 'tier', 'city', 'archetype', 'outlet_type', 'zone']],
    on='outlet_id', how='left'
)

# Rental cost by tier
rental_map = {1: 120000, 2: 75000, 3: 45000}
monthly_perf['rental_cost'] = monthly_perf['tier'].map(rental_map)

# Net margin after rental
monthly_perf['net_margin']     = round(monthly_perf['total_contribution'] - monthly_perf['rental_cost'], 2)
monthly_perf['net_margin_pct'] = round((monthly_perf['net_margin'] / monthly_perf['revenue']) * 100, 2)

# Month number for trend analysis
monthly_perf['month_num'] = pd.to_datetime(monthly_perf['month']).dt.to_period('M').apply(
    lambda x: (x.year - 2023) * 12 + x.month
)

# Sort
monthly_perf = monthly_perf.sort_values(['outlet_id', 'month']).reset_index(drop=True)

print(f"✅ monthly_performance created: {monthly_perf.shape[0]:,} rows × {monthly_perf.shape[1]} columns")
print(f"\nMonths covered: {monthly_perf['month'].nunique()}")
print(f"Outlets covered: {monthly_perf['outlet_id'].nunique()}")
print(f"\nSample KPIs (average across all outlets):")
print(f"  Avg Monthly Revenue per Outlet : ₹{monthly_perf['revenue'].mean():,.0f}")
print(f"  Avg Order Value                : ₹{monthly_perf['avg_order_value'].mean():,.0f}")
print(f"  Avg Food Cost %                : {monthly_perf['food_cost_pct'].mean():.1f}%")
print(f"  Avg Contribution Margin %      : {monthly_perf['contribution_margin_pct'].mean():.1f}%")
print(f"  Avg Net Margin %               : {monthly_perf['net_margin_pct'].mean():.1f}%")
print(f"  Avg Customer Rating            : {monthly_perf['avg_rating'].mean():.2f}")
print(f"  Avg Complaint Rate             : {monthly_perf['complaint_rate'].mean():.1f}%")
print(f"  Avg Repeat Customer %          : {monthly_perf['repeat_customer_pct'].mean():.1f}%")

✅ monthly_performance created: 1,530 rows × 39 columns

Months covered: 18
Outlets covered: 85

Sample KPIs (average across all outlets):
  Avg Monthly Revenue per Outlet : ₹214,884
  Avg Order Value                : ₹464
  Avg Food Cost %                : 31.5%
  Avg Contribution Margin %      : 43.4%
  Avg Net Margin %               : -10.9%
  Avg Customer Rating            : 3.84
  Avg Complaint Rate             : 11.5%
  Avg Repeat Customer %          : 48.5%


In [6]:
# ============================================================
# CELL 5 — GENERATE STAFF RECORDS TABLE
# Monthly staff data per outlet (1,530 rows)
# ============================================================

staff_records = []

months_list = pd.date_range(start='2023-01-01', end='2024-06-01', freq='MS').strftime('%Y-%m').tolist()

for _, outlet in outlet_master.iterrows():
    outlet_id   = outlet['outlet_id']
    archetype   = outlet['archetype']
    performance = archetypes[archetype]['performance']

    # Base staff count by outlet type
    base_staff_map = {
        'Standalone':  8,
        'Mall':        12,
        'Food Court':  6,
        'Highway':     7,
    }
    base_staff = base_staff_map[outlet['outlet_type']]

    prev_manager = False  # Track manager continuity

    for month in months_list:
        # Staff count with slight monthly variation
        total_staff = max(3, int(np.random.normal(base_staff, 1.5)))

        # Turnover — critical outlets lose more staff
        turnover_rate = max(0.02, np.random.normal(
            0.20 - performance * 0.15, 0.03
        ))
        resignations  = max(0, int(total_staff * turnover_rate))
        new_joinings  = max(0, resignations + np.random.randint(-1, 2))

        # Experience — star outlets retain experienced staff
        avg_experience = max(1, int(np.random.normal(
            6 + performance * 18, 3
        )))

        # Training hours — better outlets invest more
        training_hours = max(0, round(np.random.normal(
            2 + performance * 8, 1.5
        ), 1))

        # Manager change — more frequent in struggling/critical
        manager_change_prob = max(0.02, 0.20 - performance * 0.15)
        manager_change = 1 if np.random.random() < manager_change_prob else 0
        prev_manager   = bool(manager_change)

        # Overtime
        overtime_hours = max(0, round(np.random.normal(
            20 - performance * 10, 5
        ), 1))

        # Absenteeism
        absenteeism_pct = max(0, round(np.random.normal(
            0.15 - performance * 0.10, 0.03
        ) * 100, 1))

        staff_records.append({
            'outlet_id':             outlet_id,
            'month':                 month,
            'total_staff':           total_staff,
            'new_joinings':          new_joinings,
            'resignations':          resignations,
            'turnover_pct':          round(turnover_rate * 100, 1),
            'avg_experience_months': avg_experience,
            'training_hours':        training_hours,
            'manager_change':        manager_change,
            'overtime_hours':        overtime_hours,
            'absenteeism_pct':       absenteeism_pct,
        })

staff_df = pd.DataFrame(staff_records)

print(f"✅ staff_records created: {staff_df.shape[0]:,} rows × {staff_df.shape[1]} columns")
print(f"\nSample staff KPIs (average across all outlets):")
print(f"  Avg Staff Count        : {staff_df['total_staff'].mean():.1f}")
print(f"  Avg Turnover %         : {staff_df['turnover_pct'].mean():.1f}%")
print(f"  Avg Experience (months): {staff_df['avg_experience_months'].mean():.1f}")
print(f"  Avg Training Hours     : {staff_df['training_hours'].mean():.1f} hrs/month")
print(f"  Avg Overtime Hours     : {staff_df['overtime_hours'].mean():.1f} hrs/month")
print(f"  Avg Absenteeism %      : {staff_df['absenteeism_pct'].mean():.1f}%")
print(f"  Manager Changes Total  : {staff_df['manager_change'].sum()}")

✅ staff_records created: 1,530 rows × 11 columns

Sample staff KPIs (average across all outlets):
  Avg Staff Count        : 7.9
  Avg Turnover %         : 10.0%
  Avg Experience (months): 17.7
  Avg Training Hours     : 7.4 hrs/month
  Avg Overtime Hours     : 13.2 hrs/month
  Avg Absenteeism %      : 8.4%
  Manager Changes Total  : 141


In [7]:
# ============================================================
# CELL 6 — GENERATE CUSTOMER REVIEWS TABLE
# ~28,000 rows across 85 outlets over 18 months
# ============================================================

review_categories = ['Food Quality', 'Delivery Speed', 'Packaging', 'Quantity', 'Service', 'Value for Money']
platforms         = ['Zomato', 'Swiggy', 'Google', 'Direct']
platform_weights  = [0.45, 0.30, 0.18, 0.07]

customer_reviews = []
review_counter   = 1

for _, outlet in outlet_master.iterrows():
    outlet_id   = outlet['outlet_id']
    archetype   = outlet['archetype']
    performance = archetypes[archetype]['performance']

    # Number of reviews per outlet over 18 months
    # Star outlets get more reviews — more engaged customers
    base_reviews = {
        'star':       500,
        'stable':     350,
        'struggling': 250,
        'critical':   150,
        'new':        200,
    }
    num_reviews = max(50, int(np.random.normal(base_reviews[archetype], 40)))

    for _ in range(num_reviews):
        review_id   = f'REV{str(review_counter).zfill(7)}'
        review_counter += 1

        # Random date within 18 months
        random_days  = np.random.randint(0, 547)
        review_date  = datetime(2023, 1, 1) + timedelta(days=random_days)

        # Platform
        platform = np.random.choice(platforms, p=platform_weights)

        # Rating — performance drives rating distribution
        rating_mean = 2.0 + performance * 3.0
        rating      = round(min(5.0, max(1.0, np.random.normal(rating_mean, 0.8))))
        rating      = int(rating)

        # Review category
        # Struggling/critical outlets get more food quality and delivery complaints
        if archetype in ['critical', 'struggling']:
            cat_weights = [0.35, 0.25, 0.15, 0.10, 0.10, 0.05]
        else:
            cat_weights = [0.20, 0.20, 0.15, 0.15, 0.15, 0.15]
        review_category = np.random.choice(review_categories, p=cat_weights)

        # Sentiment score — correlated with rating
        sentiment_base  = (rating - 3) / 2
        sentiment_score = round(min(1.0, max(-1.0,
            sentiment_base + np.random.normal(0, 0.15)
        )), 2)

        # Is complaint
        is_complaint = 1 if rating <= 2 else 0

        # Was resolved — better outlets resolve more
        if is_complaint:
            resolve_prob = 0.30 + performance * 0.50
            was_resolved = 1 if np.random.random() < resolve_prob else 0
        else:
            was_resolved = 0

        # Response time — faster for star outlets
        if is_complaint:
            response_hours = max(0.5, round(np.random.normal(
                48 - performance * 36, 8
            ), 1))
        else:
            response_hours = None

        customer_reviews.append({
            'review_id':           review_id,
            'outlet_id':           outlet_id,
            'review_date':         review_date.date(),
            'platform':            platform,
            'rating':              rating,
            'review_category':     review_category,
            'sentiment_score':     sentiment_score,
            'is_complaint':        is_complaint,
            'was_resolved':        was_resolved,
            'response_time_hours': response_hours,
            'archetype':           archetype,
        })

reviews_df = pd.DataFrame(customer_reviews)

print(f"✅ customer_reviews created: {reviews_df.shape[0]:,} rows × {reviews_df.shape[1]} columns")
print(f"\nPlatform distribution:\n{reviews_df['platform'].value_counts()}")
print(f"\nRating distribution:\n{reviews_df['rating'].value_counts().sort_index()}")
print(f"\nAvg sentiment score       : {reviews_df['sentiment_score'].mean():.2f}")
print(f"Total complaints          : {reviews_df['is_complaint'].sum():,}")
print(f"Complaints resolved       : {reviews_df[reviews_df['is_complaint']==1]['was_resolved'].sum():,}")
print(f"Avg response time (hrs)   : {reviews_df['response_time_hours'].mean():.1f}")
print(f"\nReviews by archetype:\n{reviews_df.groupby('archetype')['review_id'].count()}")

✅ customer_reviews created: 28,585 rows × 11 columns

Platform distribution:
platform
Zomato    12775
Swiggy     8690
Google     5135
Direct     1985
Name: count, dtype: int64

Rating distribution:
rating
1       98
2     1077
3     5238
4    10911
5    11261
Name: count, dtype: int64

Avg sentiment score       : 0.54
Total complaints          : 1,175
Complaints resolved       : 653
Avg response time (hrs)   : 30.9

Reviews by archetype:
archetype
critical       1429
new            1501
stable         8911
star          12476
struggling     4268
Name: review_id, dtype: int64


In [8]:
# ============================================================
# CELL 7 — SAVE ALL TABLES TO CSV
# ============================================================

# Define output path
raw_path = r'D:\Projects\End-to-end projects\12. Franchise Intelligence System\Data\Raw'

# Save all tables
outlet_master.to_csv(f'{raw_path}\\outlet_master.csv', index=False)
daily_transactions.to_csv(f'{raw_path}\\daily_transactions.csv', index=False)
monthly_perf.to_csv(f'{raw_path}\\monthly_performance.csv', index=False)
staff_df.to_csv(f'{raw_path}\\staff_records.csv', index=False)
reviews_df.to_csv(f'{raw_path}\\customer_reviews.csv', index=False)

print("✅ All files saved to Data/raw folder")
print(f"\nFiles saved:")
print(f"  outlet_master.csv       — {outlet_master.shape[0]:,} rows × {outlet_master.shape[1]} cols")
print(f"  daily_transactions.csv  — {daily_transactions.shape[0]:,} rows × {daily_transactions.shape[1]} cols")
print(f"  monthly_performance.csv — {monthly_perf.shape[0]:,} rows × {monthly_perf.shape[1]} cols")
print(f"  staff_records.csv       — {staff_df.shape[0]:,} rows × {staff_df.shape[1]} cols")
print(f"  customer_reviews.csv    — {reviews_df.shape[0]:,} rows × {reviews_df.shape[1]} cols")

# Final dataset summary
total_rows = (outlet_master.shape[0] + daily_transactions.shape[0] +
              monthly_perf.shape[0] + staff_df.shape[0] + reviews_df.shape[0])

print(f"\n{'='*50}")
print(f"TOTAL DATASET SIZE: {total_rows:,} rows across 5 tables")
print(f"{'='*50}")

✅ All files saved to Data/raw folder

Files saved:
  outlet_master.csv       — 85 rows × 12 cols
  daily_transactions.csv  — 681,949 rows × 32 cols
  monthly_performance.csv — 1,530 rows × 39 cols
  staff_records.csv       — 1,530 rows × 11 cols
  customer_reviews.csv    — 28,585 rows × 11 cols

TOTAL DATASET SIZE: 713,679 rows across 5 tables
